# PopQuants · Strategy 1: Pairs Trading — AMAT × LRCX
## จากราคา 2 ตัว → Spread → Z-score → สัญญาณ → ผลตอบแทน → Tear Sheet

**Notebook สำหรับสอนแบบทีละขั้น** | ใช้ข้อมูลหุ้นสหรัฐฯ ที่ดึงจาก uSMART จริงผ่านไฟล์ CSV ในเครื่องคุณ | **Read-only / ไม่ Login / ไม่ส่ง Order**

**วิธีใช้:** เปิดใน Jupyter Lab หรือ VS Code และกด **Run All** (หรือกด Shift+Enter ทีละ Cell เพื่อสอน) โดยมีไฟล์ `~/Documents/usmart/popquants_strategy1/us_stock_adjusted_daily.csv` จากการ Backtest ก่อนหน้า

> **หลักการ:** ค่า `Spread` ในที่นี้คือความแตกต่างของ **log ราคา** ไม่ใช่ความต่างราคาดอลลาร์ตรง ๆ; เราจะเริ่มจากตัวอย่าง `ราคาหุ้น A − ราคาหุ้น B` เพื่อให้เห็นแนวคิด แล้วค่อยอธิบายเหตุผลที่โมเดลใช้ Log Spread 1:1

**ข้อจำกัด:** ข้อมูลที่เตรียมไว้เป็นราคาปรับย้อนหลัง (`right=1`) และเราต้องตรวจ Corporate Actions เสมอ; Backtest นี้สมมติ 5 bps ต่อ one-way turnover ไม่รวม Borrow Fee, Financing, Dividends และ Fill จริง

## STEP 0 — เตรียมเครื่องมือและกฎทดลอง
**สิ่งที่เราจะใช้:** `AMAT` = Applied Materials (หุ้น A) และ `LRCX` = Lam Research (หุ้น B)

ค่าต่าง ๆ ตั้งไว้ให้ตรง Strategy 1 เดิม **และสามารถทดลองปรับในห้องเรียนได้** แต่ถ้าปรับแล้วผลเปลี่ยนไป ต้องไม่อ้างว่าเป็นผล Baseline เดิม

In [ ]:
# ถ้ายังไม่ติดตั้ง ให้รันบรรทัดนี้ใน Terminal: python3 -m pip install pandas numpy matplotlib
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

STOCK_A = 'AMAT'             # หุ้นตัวแรก (A)
STOCK_B = 'LRCX'             # หุ้นตัวที่สอง (B)
UNIVERSE = ['AMAT', 'LRCX', 'KLAC', 'TER']
ROLLING_DAYS = 60           # จำนวนวันในหน้าต่างตัวอย่างของ Z-score (ตั้งตาม default เดิม)
ENTRY_Z = 2.0               # เปิดสถานะเมื่อ Z-score >= +2 หรือ <= -2
EXIT_Z = 0.5                # ปิดสถานะเมื่อ |Z-score| <= 0.5
COST_BPS = 5.0              # ต้นทุนสมมติ 5 basis points ต่อ one-way turnover
WEIGHT_EACH_LEG = 0.5       # ถือ Long 50% และ Short 50% ของเงินทุน
TRADING_DAYS_PER_YEAR = 252

print('คู่หุ้น:', STOCK_A, '/', STOCK_B)
print('Rolling days =', ROLLING_DAYS, '| Entry Z =', ENTRY_Z, '| Exit Z =', EXIT_Z, '| One-way cost =', COST_BPS, 'bps')

## STEP 1 — อ่าน Historical Data ที่ดึงจาก uSMART มาแล้ว

**ตารางที่ 1:** 1 แถว = 1 วันซื้อขาย, 1 คอลัมน์ = ราคาปิดปรับย้อนหลังของหุ้นหนึ่งตัว

- ถ้าไม่พบ CSV โปรแกรมจะบอกที่อยู่ที่ต้องตรวจ ไม่ใช้ข้อมูลแต่งขึ้นมาแทน
- ราคาใน CSV เป็น **Adjusted Close**; อย่าเทียบราคาตัวเลขกับ Unadjusted Quote โดยตรงโดยไม่ตรวจการปรับราคา

In [ ]:
# แหล่งข้อมูลใน Mac ที่ Strategy 1 เคยบันทึกไว้
price_file = Path.home() / 'Documents' / 'usmart' / 'popquants_strategy1' / 'us_stock_adjusted_daily.csv'
# เผื่อผู้สอนย้าย notebook ไปไว้ในโฟลเดอร์อื่น สามารถวาง CSV ข้าง notebook ได้เช่นกัน
if not price_file.exists():
    alternative = Path('us_stock_adjusted_daily.csv')
    if alternative.exists():
        price_file = alternative
    else:
        raise FileNotFoundError(f'ไม่พบข้อมูลราคาจริง: {price_file}. กรุณารัน Strategy 1 เดิมให้สร้าง CSV ก่อน')

raw = pd.read_csv(price_file, parse_dates=['date']).set_index('date').sort_index()
for ticker in UNIVERSE:
    raw[ticker] = pd.to_numeric(raw[ticker], errors='coerce')
prices = raw[UNIVERSE].replace([np.inf, -np.inf], np.nan).dropna(how='any')
prices = prices[~prices.index.duplicated(keep='last')]
if (prices <= 0).any().any() or len(prices) < ROLLING_DAYS + 75:
    raise ValueError('ข้อมูลราคาผิดปกติ หรือจำนวนวันไม่พอทำ Rolling Backtest')

print(f'แหล่งข้อมูล: {price_file}')
print(f'ช่วงวันที่: {prices.index.min().date()} → {prices.index.max().date()} | {len(prices)} วันซื้อขาย')
display(prices.head(7).round(3))

## STEP 2 — ทำไมต้องเป็น AMAT และ LRCX?

ทั้งคู่จำหน่ายอุปกรณ์สำหรับการผลิตเซมิคอนดักเตอร์ จึงมีปัจจัยทางเศรษฐกิจร่วมกัน เช่น **Semiconductor CapEx / วัฏจักรการลงทุนโรงงานชิป**

แต่ไม่ได้หมายความว่าราคาต้องกลับเข้าหากัน และ **ไม่ใช่ผล Pair Selection ที่ได้รับการพิสูจน์แล้ว** — เราจงใจเลือกคู่นี้เพื่อเป็น Baseline สอนกลยุทธ์

In [ ]:
# ตารางที่ 2: เปรียบเทียบราคาของสองหุ้น 5 วันแรก
comparison = prices[[STOCK_A, STOCK_B]].head(5).copy()
comparison.columns = ['ราคา A (AMAT)', 'ราคา B (LRCX)']
display(comparison.round(2))

ax = prices[[STOCK_A, STOCK_B]].plot(figsize=(11, 3.5), title='Adjusted close: AMAT & LRCX')
ax.set_ylabel('Price (adjusted)')
ax.grid(alpha=.25)
plt.show()

## STEP 3 — Spread: เริ่มต้นจากการลบราคาธรรมดา

อธิบายแบบง่ายที่สุด: **Spread = ราคาหุ้น A − ราคาหุ้น B**

ตัวอย่างสมมติ: หุ้น A = 120 ดอลลาร์, หุ้น B = 100 ดอลลาร์ ⇒ Spread = 120 − 100 = **20 ดอลลาร์**

แต่หุ้น A และ B มีระดับราคาและ Stock Split ต่างกัน ทำให้การลบราคาเป็นดอลลาร์ตรง ๆ อาจตีความผิดได้ เราจึงเปลี่ยนเป็น **Log-price Spread** ในขั้นต่อไป

In [ ]:
# ตัวอย่างตัวเลขสมมติสำหรับใช้สอนเท่านั้น (ไม่ใช่ราคาจริง)
price_a_example = 120.0
price_b_example = 100.0
simple_spread_example = price_a_example - price_b_example
print(f'สูตรแบบง่าย: Spread = ราคา A - ราคา B = {price_a_example:.2f} - {price_b_example:.2f} = {simple_spread_example:.2f} USD')

# ตารางที่ 3: ความต่างราคา 'ดอลลาร์' จากข้อมูลจริง 5 วัน
simple_spread = prices[STOCK_A] - prices[STOCK_B]
display(pd.DataFrame({'ราคา A': prices[STOCK_A], 'ราคา B': prices[STOCK_B], 'Spread แบบลบราคา':simple_spread}).head(5).round(4))

## STEP 4 — Log Spread: สมการที่ใช้จริงใน Strategy 1

\[
X_t=\ln(P_{A,t})-\ln(P_{B,t})
\]

**แปลทุกตัวแปร:**
- \(X_t\) = Spread ของวัน \(t\) (ส่วนต่างในหน่วย log ไม่ใช่ดอลลาร์)
- \(P_{A,t}\) = ราคาปรับย้อนหลังของหุ้น A ในวัน \(t\)
- \(P_{B,t}\) = ราคาปรับย้อนหลังของหุ้น B ในวัน \(t\)
- \(\ln\) = Natural Logarithm (ปุ่ม `ln` บนเครื่องคิดเลข)
- \(t\) = วันซื้อขายที่กำลังดู

**แทนค่าทีละขั้น:** นำราคาหุ้น A ไปกด `ln`, นำราคาหุ้น B ไปกด `ln` แล้วเอาผลทั้งสองมาลบกัน

วิธีนี้เป็น **1:1 log spread** ตาม Baseline ที่เราทดลอง ไม่ได้ประมาณ Hedge Ratio และไม่ได้รับประกันว่า Spread จะ Stationary

In [ ]:
# ตารางที่ 4: แสดงราคาจริง, ln(ราคา A), ln(ราคา B) และการลบกัน
log_a = np.log(prices[STOCK_A])        # ln(ราคาหุ้น AMAT)
log_b = np.log(prices[STOCK_B])        # ln(ราคาหุ้น LRCX)
spread = log_a - log_b                 # Spread = ln(AMAT) - ln(LRCX)
spread_table = pd.DataFrame({'ราคา A': prices[STOCK_A], 'ราคา B':prices[STOCK_B], 'ln(ราคา A)': log_a, 'ln(ราคา B)': log_b, 'Log Spread X': spread})
display(spread_table.head(6).round(5))
first = spread_table.iloc[0]
print(f"แทนค่าวัน {spread_table.index[0].date()}: ln({first['ราคา A']:.4f}) - ln({first['ราคา B']:.4f})")
print(f"= {first['ln(ราคา A)']:.5f} - {first['ln(ราคา B)']:.5f} = {first['Log Spread X']:.5f}")
plt.figure(figsize=(11, 3)); plt.plot(spread.index, spread, linewidth=1); plt.title('Historical log spread = ln(AMAT) - ln(LRCX)'); plt.grid(alpha=.3); plt.show()

## STEP 5 — ค่าเฉลี่ยย้อนหลัง และส่วนเบี่ยงเบนมาตรฐาน

เราจะดูว่า Spread วันนี้แตกต่างจาก Spread ปกติของ **60 วันล่าสุด** เพียงใด

\[
\mu_t=\frac{1}{N}\sum_{k=0}^{N-1}X_{t-k}
\]

- \(\mu_t\) = ค่าเฉลี่ย Spread ย้อนหลังจนถึงวัน \(t\)
- \(N\) = จำนวนวันที่ดูย้อนหลัง (ในที่นี้ **60** วัน)
- \(X_{t-k}\) = Spread ของวันที่ย้อนหลังไป \(k\) วันซื้อขาย
- \(\sum\) = เครื่องหมายบวกค่าทั้งหมด

ตัวอย่าง 3 วัน: Spread = 1, 2, 3 ⇒ ค่าเฉลี่ย = (1+2+3)/3 = **2**

**Standard deviation** (\(\sigma_t\)) วัดว่า Spread กระจายรอบค่าเฉลี่ยมากเพียงใด; ในสคริปต์ใช้ Sample Standard Deviation (`ddof=1`)

In [ ]:
# ตารางตัวอย่าง 3 วัน สำหรับอธิบายการคิดค่าเฉลี่ยและความกระจาย
example_values = pd.Series([1.0, 2.0, 3.0], index=['วัน 1','วัน 2','วัน 3'])
mean_3 = example_values.mean()
std_3 = example_values.std(ddof=1)
print('Spread สมมติ:', list(example_values), '| ค่าเฉลี่ย = (1+2+3)/3 =', mean_3)
print('Sample SD = sqrt(((1-2)^2+(2-2)^2+(3-2)^2)/(3-1)) =', round(std_3, 4))

# ตารางที่ 5: ค่าเฉลี่ยและ SD แบบ Rolling จากราคาจริง
rolling_mean = spread.rolling(ROLLING_DAYS, min_periods=ROLLING_DAYS).mean()
rolling_std = spread.rolling(ROLLING_DAYS, min_periods=ROLLING_DAYS).std(ddof=1)
roll_table = pd.DataFrame({'Spread วันนี้': spread, 'ค่าเฉลี่ยย้อนหลัง':rolling_mean, 'SD ย้อนหลัง':rolling_std})
display(roll_table.dropna().head(5).round(5))

## STEP 6 — Z-score: วันนี้เบี่ยงเบนกี่ Standard Deviations?

\[
Z_t=\frac{X_t-\mu_t}{\sigma_t}
\]

- \(Z_t\) = คะแนนความผิดปกติของ Spread วันนี้
- \(X_t\) = Log Spread วันนี้ (ขั้น 4)
- \(\mu_t\) = ค่าเฉลี่ย Spread ใน 60 วันล่าสุด (ขั้น 5)
- \(\sigma_t\) = Standard Deviation ของ Spread ใน 60 วันล่าสุด (ขั้น 5)

**คิดแบบภาษาคน:** `Z-score = (Spread วันนี้ − Spread เฉลี่ย) ÷ ความผันผวนปกติของ Spread`

ตัวอย่าง: Spread = 0.30, ค่าเฉลี่ย = 0.10, SD = 0.10 ⇒ Z = (0.30−0.10)/0.10 = **+2.0**

ถ้า Z เป็น **บวกมาก** = AMAT ดูแพงเทียบกับ LRCX ตามโมเดล ถ้า **ลบมาก** = AMAT ดูถูกเทียบกับ LRCX ตามโมเดล ทั้งหมดคือการตีความเชิงสัมพัทธ์ ไม่ใช่ Intrinsic Valuation

In [ ]:
# แทนค่าตัวอย่างสมมติด้วยตัวเลขง่าย ๆ
x_ex, mean_ex, sd_ex = .30, .10, .10
print(f'ตัวอย่างสมมติ: Z = ({x_ex:.2f} - {mean_ex:.2f}) / {sd_ex:.2f} = {(x_ex-mean_ex)/sd_ex:.2f}')

# ตารางที่ 6: คำนวณ Z-score จริงทีละตัวแปร
z = (spread - rolling_mean) / rolling_std.replace(0, np.nan)
z_table = pd.DataFrame({'Spread วันนี้ X':spread,'ค่าเฉลี่ย µ':rolling_mean,'SD σ':rolling_std,'Z-score=(X-µ)/σ':z}).dropna()
display(z_table.head(6).round(5))
last_row = z_table.iloc[-1]
print(f"ตัวอย่างราคาจริงวันที่ {z_table.index[-1].date()}: Z = ({last_row.iloc[0]:.5f} - {last_row.iloc[1]:.5f}) / {last_row.iloc[2]:.5f} = {last_row.iloc[3]:+.3f}")
ax=z.plot(figsize=(11,3.5),title='Rolling Z-score'); ax.axhline(ENTRY_Z,ls='--',c='red');ax.axhline(-ENTRY_Z,ls='--',c='green');ax.axhline(0,c='gray');ax.grid(alpha=.3);plt.show()

## STEP 7 — สร้างกฎ Long / Short / Exit ทีละวัน

กฎจาก Baseline เดิม:

- หาก **ไม่มี Position** และ Z ≥ +2 ⇒ **Short AMAT / Long LRCX** (`position = -1`)
- หาก **ไม่มี Position** และ Z ≤ −2 ⇒ **Long AMAT / Short LRCX** (`position = +1`)
- หาก **มี Position** และ |Z| ≤ 0.5 ⇒ **ปิดทั้งคู่** (`position = 0`)
- หากยังถืออยู่ และ Z กระโดดข้าม Entry อีกฝั่ง ⇒ กลับทิศ (Reversal)

**สำคัญ:** สัญญาณคำนวณจากราคาปิดวันนี้ จึงไม่อ้างว่าซื้อขายได้ที่ราคาปิดเดียวกัน Backtest ใช้ `position.shift(1)` เพื่อคำนวณ PnL ของวันถัดไป (next-bar proxy)

In [ ]:
# 1 = Long A / Short B, -1 = Short A / Long B, 0 = ไม่มีสถานะ
signal_position = pd.Series(0.0, index=prices.index)
state = 0.0
signal_events = []
for dt, score in z.items():
    previous = state
    if pd.isna(score):
        state = 0.0                      # ช่วง warmup ยังไม่เทรด
    elif state == 0.0:
        if score <= -ENTRY_Z:
            state = +1.0                 # A ถูกเมื่อเทียบกับ B: Long A, Short B
        elif score >= ENTRY_Z:
            state = -1.0                 # A แพงเมื่อเทียบกับ B: Short A, Long B
    else:
        if abs(score) <= EXIT_Z:
            state = 0.0                  # Spread เริ่มกลับมาใกล้ค่าเฉลี่ย: ปิดสถานะ
        elif state == +1.0 and score >= ENTRY_Z:
            state = -1.0                 # สัญญาณกลับฝั่ง: Reversal
        elif state == -1.0 and score <= -ENTRY_Z:
            state = +1.0                 # สัญญาณกลับฝั่ง: Reversal
    signal_position.loc[dt] = state
    if state != previous:
        signal_events.append({'วันที่เกิดสัญญาณ': dt, 'Z-score': score,
                              'สถานะเดิม': previous, 'สถานะใหม่': state,
                              'คำสั่งจำลอง': 'Long AMAT / Short LRCX' if state==1 else 'Short AMAT / Long LRCX' if state==-1 else 'Exit'})

signal_events_df = pd.DataFrame(signal_events)
print('ตารางที่ 7: ตัวอย่างสัญญาณแรก ๆ (ยังไม่ใช่รายการส่ง Order จริง)')
display(signal_events_df.head(10).round(3))
print('จำนวนครั้งที่สถานะเปลี่ยน:',len(signal_events_df))

## STEP 8 — Position ที่ใช้คำนวณผลตอบแทน ไม่ใช่ Signal ณ วันเดียวกัน

\[
	ext{Position used on day }t = 	ext{Signal decided on day }t-1
\]

**ภาษาไทย:** ถ้าเห็นสัญญาณตอนตลาดปิดวันจันทร์ จะยังไม่นำสัญญาณนั้นไปคูณผลตอบแทนที่เกิดขึ้นก่อนปิดตลาดวันจันทร์

นี่เป็นวิธีป้องกันการเอาข้อมูลอนาคตมาใช้แบบง่าย ๆ แต่ยัง **ไม่เทียบเท่าการจำลองราคาซื้อขายจริงในวันถัดไป**

In [ ]:
# ตารางที่ 8: วันที่เห็นสัญญาณ กับวันที่นำ Position มาใช้ทำ PnL
executed_position = signal_position.shift(1).fillna(0.0)
position_table = pd.DataFrame({'Z-score ณ วันปิด':z,'สัญญาณที่ปิดวันนี้':signal_position,
                               'Position สำหรับคำนวณ PnL วันนี้':executed_position})
# แสดงรอบจุดที่สัญญาณเริ่มเปลี่ยน
first_event = signal_events_df['วันที่เกิดสัญญาณ'].iloc[0] if not signal_events_df.empty else prices.index[ROLLING_DAYS]
loc = prices.index.get_loc(first_event)
display(position_table.iloc[max(0,loc-2):loc+5].round(3))

## STEP 9 — Daily Return ของหุ้น A และ B

\[
r_{A,t}=\frac{P_{A,t}}{P_{A,t-1}}-1
\]

- \(r_{A,t}\) = ผลตอบแทนหุ้น A ในวัน \(t\)
- \(P_{A,t}\) = ราคาหุ้น A วันนี้
- \(P_{A,t-1}\) = ราคาหุ้น A เมื่อวันซื้อขายก่อน

**ภาษาไทย:** `Return วันนี้ = ราคาวันนี้ ÷ ราคาเมื่อวาน − 1`

ตัวอย่าง: เมื่อวาน 100, วันนี้ 103 ⇒ Return = 103/100−1 = **3%**

In [ ]:
print('ตัวอย่างสมมติ: Return = 103/100 - 1 =', f'{(103/100-1)*100:.1f}%')
r_a = prices[STOCK_A].pct_change().fillna(0.0) # ผลตอบแทน AMAT
r_b = prices[STOCK_B].pct_change().fillna(0.0) # ผลตอบแทน LRCX
returns_table = pd.DataFrame({'ราคาหุ้น A':prices[STOCK_A], 'ราคา A เปลี่ยน %':r_a*100,
                              'ราคาหุ้น B':prices[STOCK_B], 'ราคา B เปลี่ยน %':r_b*100})
print('ตารางที่ 9: คำนวณผลตอบแทนหุ้นรายวัน')
display(returns_table.iloc[1:7].round(3))

## STEP 10 — Gross Return: กำไรขาดทุนก่อนต้นทุน

\[
R^{gross}_t=\text{Position}_t \times 0.5\times(r_{A,t}-r_{B,t})
\]

**อธิบายทุก term:**
- \(R^{gross}_t\) = ผลตอบแทนของ **พอร์ต Long/Short** ก่อนหักต้นทุนในวัน \(t\)
- `Position` = **+1** เมื่อ Long AMAT/Short LRCX; **−1** เมื่อ Short AMAT/Long LRCX; **0** เมื่อไม่เทรด
- `0.5` = ให้น้ำหนักหุ้นแต่ละขา 50% ของทุน (Gross Exposure รวม 100%)
- \(r_A-r_B\) = ผลต่างผลตอบแทนรายวันของหุ้น A กับ B

ตัวอย่าง: Position = +1, AMAT +3%, LRCX +1% ⇒ Gross Return = 1 × 0.5 × (3% − 1%) = **+1% ของทุน**

In [ ]:
example_pos=1; example_ra=.03; example_rb=.01
print(f'ตัวอย่างสมมติ: Gross = {example_pos} × 0.5 × ({example_ra:.0%} - {example_rb:.0%}) = {example_pos*.5*(example_ra-example_rb):.2%}')
# ตารางที่ 10: นำ Position ที่เลื่อนหนึ่งวันแล้ว มาคูณ Return จริง
gross_return = executed_position * WEIGHT_EACH_LEG * (r_a - r_b)
gross_table = pd.DataFrame({'Position ใช้วันนี้':executed_position,'AMAT return':r_a,'LRCX return':r_b,'Gross return':gross_return})
display(gross_table.loc[gross_table['Position ใช้วันนี้']!=0].head(7).round(5))

## STEP 11 — Turnover และ Net Return: ต้นทุน 5 bps คืออะไร?

\[
\text{Turnover}_t=|\text{Position}_t-\text{Position}_{t-1}|
\]
\[
R^{net}_t=R^{gross}_t-\text{Turnover}_t\times\frac{5}{10,000}
\]

- **Turnover** = ปริมาณการปรับสถานะเทียบกับทุน; `0→+1` = 100% gross turnover, `+1→−1` = 200%
- **5 bps** = 5 ÷ 10,000 = 0.0005 = **0.05%** ต่อหนึ่งหน่วย Turnover
- **Net Return** = Gross Return หลังหักต้นทุนสมมติ

**ตัวอย่าง:** Gross = 1%, Turnover = 1 ⇒ ต้นทุน 1 × 0.0005 = 0.05% ⇒ Net = **0.95%**

**คำเตือน:** ยังไม่รวม Stock Borrow Fee, Financing, Dividends และ Fill/Slippage จริง

In [ ]:
turnover = executed_position.diff().abs().fillna(executed_position.abs())
cost_rate = COST_BPS / 10000.0
trading_cost = turnover * cost_rate
net_return = gross_return - trading_cost
print(f'ตัวอย่างสมมติ: Net = 1.00% - (1 × {COST_BPS:.0f}/10000) = {(.01 - .0005):.2%}')
print('ตารางที่ 11: ต้นทุนกับผลตอบแทนในวันที่มีการปรับ Position')
net_table = pd.DataFrame({'Position':executed_position,'Turnover':turnover,'Gross %':100*gross_return,'Cost %':100*trading_cost,'Net %':100*net_return})
display(net_table.loc[net_table['Turnover']>0].head(10).round(4))

## STEP 12 — Equity Curve: สะสมผลตอบแทนอย่างไร?

\[
\text{Equity}_t=\prod_{s=1}^{t}(1+R_s)
\]

- **Equity** = มูลค่าทุนแบบเริ่มต้น 1.00 (หรือ 100%)
- \(R_s\) = Return ของวัน \(s\)
- \(\prod\) = การคูณสะสมจากวันแรกถึงวันนี้

ตัวอย่างผลตอบแทน +10% และ −5% ⇒ Equity = 1.10 × 0.95 = **1.045** หรือกำไรสะสม **4.5%** ไม่ใช่ 5% แบบบวกตัวเลขธรรมดา

In [ ]:
print('ตัวอย่างสมมติ: Equity = (1+0.10) × (1-0.05) =',round(1.1*.95,4))
first_valid = z.first_valid_index()
if first_valid is None: raise RuntimeError('ไม่มี Z-score ที่ถูกต้อง')
start_idx = prices.index.get_loc(first_valid)
if len(prices) - start_idx < 60: raise RuntimeError('ช่วง Backtest หลัง warm-up สั้นเกินไป')
backtest = pd.DataFrame({'spread':spread,'z':z,'signal_position':signal_position,
                         'executed_position':executed_position,'gross_return':gross_return,
                         'turnover':turnover,'cost':trading_cost,'net_return':net_return},index=prices.index).iloc[start_idx:].copy()
backtest['gross_equity'] = (1 + backtest['gross_return']).cumprod()
backtest['equity'] = (1 + backtest['net_return']).cumprod()
backtest['drawdown'] = backtest['equity'] / backtest['equity'].cummax() - 1
print('ตารางที่ 12: Equity รายวันจาก Gross/Net Return')
display(backtest[['gross_return','net_return','gross_equity','equity','drawdown']].head(7).round(5))
ax=backtest[['gross_equity','equity']].plot(figsize=(12,4),title='AMAT/LRCX: Gross vs Net Equity')
ax.set_ylabel('Starting equity = 1.0');ax.grid(alpha=.3);plt.show()

## STEP 13 — Drawdown: จากจุดสูงสุดเคยลงไปเท่าไร?

\[
\text{Drawdown}_t=\frac{\text{Equity}_t}{\max(\text{Equity}_1,\ldots,\text{Equity}_t)}-1
\]

- **Equity วันนี้** = มูลค่าพอร์ตสะสม ณ วันนี้
- **Running peak** = ค่าสูงสุดที่ Equity เคยทำได้ก่อน/ถึงวันนี้
- **Drawdown** = เปอร์เซ็นต์ที่ลดลงจากยอดสูงสุด

ตัวอย่าง: เคยขึ้นไปที่ 1.20 แต่วันนี้เหลือ 1.08 ⇒ Drawdown = 1.08/1.20−1 = **−10%**

In [ ]:
print('ตัวอย่างสมมติ: Drawdown = 1.08/1.20 - 1 =', f'{(1.08/1.2-1):.1%}')
print('ตารางที่ 13: Equity, จุดสูงสุดก่อนหน้า, Drawdown')
backtest['running_peak'] = backtest['equity'].cummax()
display(backtest[['equity','running_peak','drawdown']].head(7).round(5))
ax=backtest['drawdown'].mul(100).plot(figsize=(12,3.5),title='Underwater: Net Drawdown (%)');ax.fill_between(backtest.index,backtest['drawdown']*100,0,alpha=.25);ax.grid(alpha=.3);plt.show()

## STEP 14 — Sharpe, CAGR, Volatility: อ่านผลลัพธ์อย่างไร?

**Sharpe (risk-free = 0 ตาม Baseline นี้)**
\[
\text{Sharpe}=\frac{\text{ค่าเฉลี่ย Return รายวัน}}{\text{SD ของ Return รายวัน}}\sqrt{252}
\]
- ตัวเศษ = ผลตอบแทนเฉลี่ยรายวัน
- ตัวส่วน = ความผันผวนของผลตอบแทนรายวัน
- \(\sqrt{252}\) = ปรับหน่วยให้เป็นต่อปี โดยประมาณ 252 วันซื้อขาย

**CAGR**
\[
\text{CAGR}=\text{Final Equity}^{252/N}-1
\]
- **Final Equity** = มูลค่าพอร์ตสุดท้าย (เริ่มจาก 1)
- **N** = จำนวนวันซื้อขายในช่วง Backtest

Sharpe สูงไม่ได้แปลว่ามีกำไรแน่นอน ต้องดู Drawdown, Turnover และต้นทุนด้วย

In [ ]:
# คำนวณ Metrics ในรูปแบบเดียวกับ Strategy 1 เดิม
ret = backtest['net_return']; n=len(ret); years=n/TRADING_DAYS_PER_YEAR
sd=ret.std(ddof=1)
downside_sd=ret[ret<0].std(ddof=1)
final_equity=float(backtest['equity'].iloc[-1])
cagr=final_equity**(1/years)-1
sharpe=(ret.mean()/sd)*np.sqrt(TRADING_DAYS_PER_YEAR) if sd>0 else np.nan
sortino=(ret.mean()/downside_sd)*np.sqrt(TRADING_DAYS_PER_YEAR) if downside_sd>0 else np.nan
mdd=float(backtest['drawdown'].min())
metrics = pd.Series({
    'Start':str(backtest.index.min().date()),'End':str(backtest.index.max().date()),
    'Trading days':n,'Total Return %':100*(final_equity-1),'CAGR %':100*cagr,
    'Ann. volatility %':100*sd*np.sqrt(TRADING_DAYS_PER_YEAR),
    'Sharpe (rf=0)':sharpe,'Sortino (rf=0)':sortino,'Max drawdown %':100*mdd,
    'Calmar':cagr/abs(mdd) if mdd<0 else np.nan,
    'Daily win-rate %':100*(ret>0).mean(),'Active days %':100*(backtest['executed_position']!=0).mean(),
    'Signal events':sum(pd.to_datetime(signal_events_df['วันที่เกิดสัญญาณ']) >= backtest.index.min()) if not signal_events_df.empty else 0,
})
print('ตารางที่ 14: ผล Backtest สรุปจากข้อมูลจริง')
display(metrics.to_frame('Result').round(3))

## STEP 15 — สัญญาณเข้าออกและรายงานผลแบบ PyFolio

แสดงผล **Z-score + Entry/Exit** และ **Return Heatmap** เพื่อให้ผู้เรียนมองเห็นว่าโมเดลตัดสินใจเมื่อไร และผลตอบแทนอยู่ช่วงไหน

สัญญาณในตารางเป็นการตัดสินใจจำลองจากราคา ณ วันนั้น ไม่ใช่ Order ที่ส่งเข้า Broker

In [ ]:
# กราฟสัญญาณจริงจากช่วง Backtest
fig, ax=plt.subplots(figsize=(12,4))
ax.plot(backtest.index,backtest['z'],label='Z-score',lw=1.1)
ax.axhline(ENTRY_Z,c='red',ls='--',lw=1);ax.axhline(-ENTRY_Z,c='green',ls='--',lw=1)
ax.axhline(EXIT_Z,c='gray',ls=':',lw=.8);ax.axhline(-EXIT_Z,c='gray',ls=':',lw=.8)
colors={'Long AMAT / Short LRCX':'green','Short AMAT / Long LRCX':'red','Exit':'gray'}
markers={'Long AMAT / Short LRCX':'^','Short AMAT / Long LRCX':'v','Exit':'x'}
for action in colors:
    if signal_events_df.empty: continue
    sel=signal_events_df[signal_events_df['คำสั่งจำลอง']==action]
    sel=sel[pd.to_datetime(sel['วันที่เกิดสัญญาณ']).isin(backtest.index)]
    if len(sel): ax.scatter(sel['วันที่เกิดสัญญาณ'],sel['Z-score'],label=action,color=colors[action],marker=markers[action],s=36,zorder=3)
ax.set_title('Signal dates vs Z-score (positions used from next bar)');ax.legend(loc='upper left',fontsize=8);ax.grid(alpha=.25);plt.show()

print('ตารางที่ 15: สัญญาณที่อยู่ในช่วงผล Backtest')
display(signal_events_df.loc[pd.to_datetime(signal_events_df['วันที่เกิดสัญญาณ'])>=backtest.index.min()].head(25).round(3))

## STEP 16 — Monthly Returns Heatmap และ Benchmark (Equal-weighted Buy & Hold)

**Monthly Return** ไม่ใช่การนำ Daily Return มาบวกตรง ๆ แต่ใช้การคูณสะสม `∏(1 + daily_return) − 1`

**Benchmark:** ซื้อ AMAT, LRCX, KLAC, TER หุ้นละ **25% ของทุนตั้งต้น** แล้วถือไว้โดยไม่ Rebalance (Buy & Hold) — ไม่ใช่ Long/Short Neutral; ใช้เทียบการลงทุนอีกแบบ ไม่ใช่ Risk-matched Benchmark

In [ ]:
# คำนวณผลตอบแทนรายเดือนแบบ Compound
monthly=backtest['net_return'].resample('ME').apply(lambda x: (1+x).prod()-1)
month_df=pd.DataFrame({'year':monthly.index.year,'month':monthly.index.month,'return_pct':monthly.values*100})
monthly_grid=month_df.pivot(index='year',columns='month',values='return_pct').reindex(columns=range(1,13))
print('ตารางที่ 16: Monthly Return (%)')
display(monthly_grid.round(2))
fig,ax=plt.subplots(figsize=(12,3.4))
values=monthly_grid.to_numpy(float)
im=ax.imshow(values,aspect='auto',cmap='RdYlGn',vmin=-max(1,np.nanmax(abs(values))),vmax=max(1,np.nanmax(abs(values))))
ax.set_xticks(range(12),['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']);ax.set_yticks(range(len(monthly_grid)),monthly_grid.index)
for i in range(values.shape[0]):
  for j in range(values.shape[1]):
    if np.isfinite(values[i,j]): ax.text(j,i,f'{values[i,j]:.1f}',ha='center',va='center',fontsize=8)
ax.set_title('Strategy 1: Monthly returns (%)');fig.colorbar(im,ax=ax);plt.show()

# Equal weighted buy-and-hold, แต่ละตัวเริ่มต้น 25% ของทุน แล้วปล่อยน้ำหนักลอยไปตามราคา
benchmark_prices = prices.loc[backtest.index, UNIVERSE]
initial = benchmark_prices.iloc[0]
benchmark_equity=(benchmark_prices / initial).mul(0.25).sum(axis=1)
benchmark_return=benchmark_equity.pct_change().fillna(0)
print('Benchmark เงินต้น 1 หน่วย →',round(benchmark_equity.iloc[-1],4),'| Total Return =',f'{(benchmark_equity.iloc[-1]-1)*100:.2f}%')
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(backtest.index,backtest['equity'],label='Strategy 1 (net)')
ax.plot(benchmark_equity.index,benchmark_equity,label='Equal-weight buy & hold (4 stocks)')
ax.set_title('Strategy vs unequal-risk benchmark');ax.legend();ax.grid(alpha=.3);plt.show()

## STEP 17 — ส่งออกตารางสำหรับนำไปสอน และตรวจ Baseline

**ข้อมูลจะถูกบันทึกเป็นไฟล์ใหม่** โดยไม่ทับ CSV ที่โหลดมาจาก uSMART

หากใช้ Dataset และ Parameter เดียวกับการรัน Strategy 1 ก่อนหน้า ตัวเลขควรใกล้เคียงหรือเท่ากัน (ขึ้นกับช่วงไฟล์ข้อมูล) ถ้าไม่ตรงให้ตรวจ `ROLLING_DAYS`, `ENTRY_Z`, `EXIT_Z`, ราคาและช่วงวันที่ก่อน ไม่แก้ตัวเลขให้เหมือนกันแบบฝืน ๆ

In [ ]:
outdir=Path.home()/'Documents'/'usmart'/'popquants_strategy1'/'teaching_notebook_outputs'
outdir.mkdir(parents=True,exist_ok=True)
backtest.to_csv(outdir/'step_by_step_backtest.csv')
signal_events_df.to_csv(outdir/'step_by_step_signals.csv',index=False)
metrics.to_frame('Result').to_csv(outdir/'step_by_step_metrics.csv')
monthly_grid.to_csv(outdir/'monthly_return_heatmap.csv')
print('บันทึกผลที่:',outdir)
for p in sorted(outdir.iterdir()): print(' -',p.name)
print('\nสรุป: กลยุทธ์นี้มีความสัมพันธ์เชิงอุตสาหกรรมเป็นสมมติฐานตั้งต้น แต่ต้องประเมิน Edge จาก Out-of-Sample + Costs เสมอ')

## คำถามสำหรับผู้เรียน (อภิปรายท้ายบท)

1. AMAT กับ LRCX ทำธุรกิจที่เชื่อมโยงกันอย่างไร และมีความเสี่ยงพื้นฐานที่แตกต่างกันตรงไหน?
2. ทำไม `ราคา A − ราคา B` กับ `ln(ราคา A) − ln(ราคา B)` ให้ Spread ที่ตีความต่างกัน?
3. เมื่อ Z-score = +2 เรากำลังพิสูจน์ว่า AMAT แพงจริง หรือเพียงตั้งสมมติฐาน?
4. ถ้าเปลี่ยน Entry Z, Window และ Transaction Costs ผล Sharpe จะเปลี่ยนแค่ไหน? จะป้องกัน Overfitting อย่างไร?
5. ทำไมผลของ Equal-weighted Long-only ไม่ควรนำมาเทียบกับ Dollar-neutral Long/Short โดยไม่ดู Risk Exposure?

**คำเตือนก่อนใช้จริง:** ต้องทดสอบด้วยช่วง Out-of-Sample, ราคาที่เทรดได้จริง, Borrow Fee, Short Constraints, Dividends, Corporate Actions และ Execution Timing